# SPE10: published Darcy and Brinkman spaces

The macrogrid is 6×11 squares, pressure is local Q1, and each face has a continuous P1 trace on 32 segments. The article does not report its local subdivision count; this study varies that count independently. The conforming Q3 comparison uses the article's 768×1408 grid and 9,738,625 unknowns. It is assembled separately in PyMHM with shared basis kernels, not in an independent code.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.cartesian_darcy import define_cartesian_darcy, recover_cartesian_darcy

from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from pymhm import CartesianMacroMesh, FaceSpace, SkeletonSpace

OUTPUT = ROOT / "examples/results/spe10"
mesh = CartesianMacroMesh(2, 2)
skeleton = SkeletonSpace(
    mesh, tuple((FaceSpace.uniform(1, 2, continuous=True) for _ in mesh.faces))
)
solution_definition = define_cartesian_darcy(
    mesh,
    skeleton=skeleton,
    degree=2,
    local_refinement=3,
    dirichlet=lambda x: 1 + x[:, 0] - 2 * x[:, 1],
)
solution_system = assemble(solution_definition.problem)
solution = recover_cartesian_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(lambda x: 1 + x[:, 0] - 2 * x[:, 1]) < 1e-11
print("Native Q2 affine patch verified")


## Checks against the archived published-size runs

The checks below cover file identity, reported spaces, global dimension and conservation. They do not replace the assembly and solve commands in the case documentation. Published curve values are raster samples with a stated uncertainty, not exact field coefficients.


In [ ]:
rows = [json.loads(p.read_text()) for p in OUTPUT.glob("darcy-q1-r*-s32.json")]
aligned = sorted((r for r in rows if r["coefficient_pixels_aligned"]), key=lambda r:r["local_refinement"][0])
assert len(aligned) >= 5
for row in aligned:
    assert row["macro_cells"] == 66 and row["degree"] == 1
    assert row["skeleton_degree"] == 1 and row["skeleton_continuous"]
    assert row["free_trace_plus_retained"] == 4257
    assert row["macro_balance_max"] < 1e-11
    assert hashlib.sha256((OUTPUT / row["archive"]).read_bytes()).hexdigest() == row["sha256"]
    print(row["local_refinement"][0], -row["exterior_flux_bottom_right_top_left"][0], row["macro_balance_max"])
reference = json.loads((OUTPUT / "reference-q3-768x1408-order5.json").read_text())
assert reference["degrees_of_freedom"] == 9738625
assert reference["cells"] == 1081344
assert reference["free_residual_relative"] < 1e-10
assert hashlib.sha256((OUTPUT/reference["archive"]).read_bytes()).hexdigest() == reference["sha256"]
comparison = json.loads((OUTPUT / "darcy-comparison.json").read_text())
print("Raster pressure uncertainty:", comparison["pressure_digitization_uncertainty"])


In [ ]:
for name in ("l07-figure-6.png", "l07-figure-7-8.png"):
    display(Image(filename=str(ROOT / "docs/figures/reservoir-papers" / name)))
for name in ("darcy-fields", "darcy-reference-fields", "darcy-profiles", "darcy-refinement"):
    display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


The expected flow follows the connected permeable channels and preserves total inflow/outflow. Solid refinement curves use material-aligned cells; dashed curves use cells crossing permeability jumps, with exact intersection quadrature. Exact integration does not add gradient jumps to the Q1 space, so these grid families are not a single monotone convergence sequence. All spatial panels show the actual macrogrid and profiles retain independent one-sided values.

Source: [Paredes, Valentin and Versieux, Section 5.2 and Figures 5–8](https://doi.org/10.1016/j.cam.2023.115415). See the SPE10 case page for the separately specified 2017 Brinkman configuration and its verification.


## Darcy flux and independent classical references

The layer-36 flux figure in Barrenechea et al. (2026) uses triangular P2 local pressure, constant traces and RT2 reconstruction. This differs from the fixed quadrilateral Q1/continuous-P1 MHM computation. The article image is a qualitative field comparison; no vector arrays or numerical flux profile are supplied.

The computed comparisons use the same material and boundary data. Q3 is a separate conforming assembly using PyMHM kernels. MSL_CG is a native conforming P1 assembly; NeoPZ is a native mixed RT0/P0 assembly. Their mesh refinements quantify the uncertainty of each numerical reference. These fields are not exact solutions. Reported L2 differences integrate vector fields on mesh intersections, whereas the maps show identical material-pixel center samples. Every panel includes the 66-square macro partition.

Signed component maps use a blue–white–red palette and a symmetric asinh color scale. Colorbar labels retain physical flux values; MHM and reference use the same scale, while signed differences use their own scale. Zero is white, and no values are clipped or spatially smoothed.


In [ ]:
study = json.loads((OUTPUT / "darcy-flux-comparison.json").read_text())
for record in (study["mhm"], *study["references"].values()):
    archive = OUTPUT / record["archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == record["sha256"]
    with np.load(archive) as data:
        assert data["flux"].shape == (60, 220, 2)
        assert np.isfinite(data["flux"]).all()
for method, sequence in study["refinement"].items():
    assert len(sequence) >= 4
    print(method)
    for row in sequence:
        print(row["cells"], row["inlet_flux"], row.get("flux_relative"))
print("Integrated fixed-MHM comparisons:")
for row in study["mhm_differences"]:
    print(row["reference"], row["pressure_relative"], row["flux_relative"])
display(Image(filename=str(ROOT / "docs/figures/reservoir-papers/l09-figure-6.png")))
for name in ("darcy-flux-q3", "darcy-flux-q3-components", "darcy-flux-classical-references", "darcy-flux-msl", "darcy-flux-neopz", "darcy-flux-reference-refinement"):
    display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


## Brinkman: the 2017 P3/P3 configuration

The main case uses layer 1, 264 crisscross macrotriangles, 100 fine triangles per macro, local P3/P3 USFEM, and vector DG P1 traces on ten segments per face. Slip walls and zero top pseudotraction follow the article. The minimum-eigenvalue parameter is evaluated pointwise on material pixels; the paper does not specify this spatial convention or its numerical inverse constant. Matching the printed spaces therefore does not establish an identical historical calculation. Its MHM colorbar maxima are 9.090 (velocity magnitude) and 478.9 (pressure); the computed nodal maxima remain different and are printed below.


In [ ]:
flow = json.loads((OUTPUT / "flow-layer1-n6x11-p3-r10-s10-q5-pointwise-2017.json").read_text())
assert flow["macro_cells"] == 264
assert flow["velocity_degree"] == flow["pressure_degree"] == 3
assert flow["local_refinement"]**2 == 100
assert flow["skeleton_degree"] == 1 and flow["skeleton_segments"] == 10
assert not flow["skeleton_continuous"]
assert flow["total_skeleton_unknowns"] == 16520
assert flow["stabilization"] == "pointwise-2017"
assert hashlib.sha256((OUTPUT / flow["archive"]).read_bytes()).hexdigest() == flow["sha256"]
check = json.loads((OUTPUT / "flow-quadrature.json").read_text())["rows"][0]
assert check["velocity"]["relative_to_higher"] < 1e-9
assert check["pressure"]["relative_to_higher"] < 1e-9
print("Computed nodal maxima:", flow["velocity_magnitude_nodal_max"], flow["pressure_nodal_max"])
print("Order-5/order-8 integrated relative differences:", check["velocity"]["relative_to_higher"], check["pressure"]["relative_to_higher"])
for name in ("brinkman-layer1", "brinkman-layer1-profiles"):
    display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


The integration study changes quadrature only, leaving the physical finite-element spaces fixed. The expected channel flow and pressure barrier are recovered, but the remaining discrepancy with the published extrema is not hidden or called an exact reproduction. Independent DOLFINx/UFL checks of the three residual-parameter conventions are recorded in `examples/results/usfem-stabilization-operators.json`. They verify matrices and loads, not the historical parameter selection. Source: [Araya et al. (2017), Figures 18–20](https://doi.org/10.1016/j.cma.2017.05.027).


## Classical conforming Taylor–Hood reference

The P2/P1 reference uses native DOLFINx/UFL and PETSc/MUMPS, with no residual stabilization. The layer, operator, units and mixed boundary conditions match the MHM calculation. Pixel-aligned meshes retain exactly the same material. The top traction fixes the pressure constant, so no pressure shift is introduced. This additional classical baseline is distinct from the article's USFEM reference.

Successive-reference differences use exact quadrature on nested fine triangles. MHM/reference L2 differences are integrated over the broken MHM fine triangles, with an independent quadrature-order study. The finest reference remains a numerical approximation; its own refinement change must be read alongside the MHM difference.

In [ ]:
th = sorted((json.loads(p.read_text()) for p in OUTPUT.glob("taylor-hood-[0-9]*x[0-9]*.json")), key=lambda r:r["unknowns"])
assert len(th) >= 4
for row in th:
    assert row["velocity_degree"] == 2 and row["pressure_degree"] == 1
    assert row["mesh_shape"][0] % 60 == row["mesh_shape"][1] % 220 == 0
    assert row["original_free_residual_relative"] < 1e-10
    assert abs(row["exterior_balance"]) < 1e-6
    assert row["layer_sha256"] == hashlib.sha256((OUTPUT / "layer-1.npz").read_bytes()).hexdigest()
    assert row["sha256"] == hashlib.sha256((OUTPUT / row["archive"]).read_bytes()).hexdigest()
    print(row["mesh_shape"], row["unknowns"], row.get("successive_difference", {}))
nx, ny = th[-1]["mesh_shape"]
norms = json.loads((OUTPUT / f"taylor-hood-mhm-{nx}x{ny}.json").read_text())
assert norms["reference_coefficients_sha256"] == th[-1]["coefficient_sha256"]
assert norms["mhm_sha256"] == hashlib.sha256((OUTPUT / norms["mhm"]).read_bytes()).hexdigest()
ordered = sorted(norms["norms"], key=lambda r:r["quadrature_order"])
for variable in ("velocity", "pressure"):
    key = variable + "_relative"
    final = ordered[-1][key]
    quadrature_change = abs(ordered[-2][key] / final - 1)
    reference_change = th[-1]["successive_difference"][key]
    print(variable, "MHM/reference L2:", final, "quadrature relative change:", quadrature_change,
          "successive-reference L2:", reference_change,
          "reference change / MHM difference:", reference_change/final)
for name in ("fields", "refinement", "profiles", "profiles-detail"):
    display(Image(filename=str(ROOT / "docs/figures/spe10" / f"brinkman-taylor-hood-{name}.png")))